# Convolution using only einops, einsum

In [1]:
#! pip install einops einsum

In [69]:
import torch
from torch import randn
#from einsum import einsum
from torch import einsum
from einops import rearrange, repeat, reduce
from torch.nn import Unfold, Module, Parameter

import itertools
import math

# import kaiming_uniform_ as kaiming_uniform
from torch.nn.init import kaiming_uniform_

In [118]:
class EinConv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1,padding=0):
        super(EinConv2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels

        self.kernel_size = kernel_size if type(kernel_size) == tuple else (kernel_size, kernel_size)
        self.stride = stride if type(stride) == tuple else (stride, stride)
        self.dilation = dilation if type(dilation) == tuple else (dilation, dilation)
        self.padding = padding if type(padding) == tuple else (padding, padding)

        self._init_kernel()

    def _init_kernel(self):
        kernel_size = tuple(itertools.chain((self.out_channels,self.in_channels), self.kernel_size))
        self.weight = torch.randn(kernel_size)
        self.weight = Parameter(torch.empty(
                (self.out_channels, self.in_channels, *self.kernel_size)))
        torch.manual_seed(42)
        kaiming_uniform_(self.weight, a=math.sqrt(5))

    def _unfold(self,x):
        B,c_in,h_in,w_in = x.shape
        assert self.in_channels == c_in
        h_out = math.floor(((h_in + 2*self.padding[0] - self.dilation[0]*(self.kernel_size[0]-1) - 1)/self.stride[0])+1)
        w_out = math.floor(((w_in + 2*self.padding[1] - self.dilation[1]*(self.kernel_size[1]-1) - 1)/self.stride[1])+1) 
        
        # unfold to create patches
        x_hat = torch.nn.functional.unfold(x, self.kernel_size, dilation=self.dilation, stride=self.stride) #Bx(CxKxK)x(H_out,W_out)

        # reshape
        x_hat = rearrange(x_hat, 'b (c k l) (h w) -> b c h w k l', c=self.in_channels,k=self.kernel_size[0],l=self.kernel_size[1],h=h_out,w=w_out)
        return x_hat
    
    def _conv(self,x):
        x_conv_k = torch.einsum('bcghkl, fckl -> bfgh',x,self.weight)
        return x_conv_k

    def forward(self,x):
        x = self._unfold(x)
        x = self._conv(x)
        return x

## Validation with Native Torch conv2d

In [146]:
B = torch.randint(1,10,(1,))
c = torch.randint(1,10,(1,))
h = torch.randint(5,20,(1,))
w = torch.randint(5,20,(1,))
x = torch.randn(B, c ,h ,w)

print(x.shape)

B,c_in,h_in,w_in = map(int,x.shape)
c_out = 10 # number of kernels
ein_cov2d = EinConv2d(in_channels=c_in, out_channels=c_out, kernel_size=(3,3))
z = ein_cov2d(x)

#print(ein_cov2d.weight)

torch.manual_seed(42)
conv2d = torch.nn.Conv2d(in_channels=c_in, out_channels=c_out, kernel_size=(3,3), stride=1, padding=0, dilation=1, groups=1, bias=False)
torch.allclose(conv2d(x), z, atol=1e-5)

torch.Size([7, 6, 10, 19])


True